# EXP-30: Interfacial Alanine Scanning via Alchemical FEP
**15 BPTI mutations × 2 legs (complex + free) = 30 FEP campaigns**

- **PASS:** R > 0.7 AND RMSE < 2.0 kcal/mol AND ≥80% correct sign
- **MARGINAL:** R > 0.5 AND RMSE < 3.0 | **FAIL:** otherwise
- **Most GPU-intensive experiment (~1200 ns total FEP)**

In [ ]:
# Install OpenMM + all scientific dependencies (pip — OpenCL platform)
!pip install openmm mdtraj 'pymbar==4.0.3' mdanalysis
!pip install netCDF4 git+https://github.com/choderalab/openmmtools.git

# mpiplus stub — not on PyPI; minimal shim for imports
import os as _os, sys as _sys
_sp = _os.path.join(_sys.prefix, 'lib',
    f'python{_sys.version_info.major}.{_sys.version_info.minor}',
    'site-packages', 'mpiplus')
_os.makedirs(_sp, exist_ok=True)
with open(_os.path.join(_sp, '__init__.py'), 'w') as _f:
    _f.write(
        'class delayed_termination:\n'
        '    def __init__(self, func=None):\n'
        '        self._func = func\n'
        '    def __enter__(self): return self\n'
        '    def __exit__(self, *a): pass\n'
        '    def __call__(self, *args, **kwargs):\n'
        '        if self._func is not None: return self._func(*args, **kwargs)\n'
        '        if len(args) == 1 and callable(args[0]): return args[0]\n'
        '        return self\n'
        'def get_mpicomm(): return None\n'
        'def run_single_node(rank=0, broadcast_result=False):\n'
        '    def decorator(func): return func\n'
        '    return decorator\n'
        'def on_single_node(rank=0, broadcast_result=False, sync_nodes=False):\n'
        '    def decorator(func): return func\n'
        '    return decorator\n'
        'def distribute(func, seq, *args, send_results_to=None, sync_nodes=False, group_nodes=None):\n'
        '    results = [func(item, *args) for item in seq]\n'
        '    return list(zip(*results)) if results and isinstance(results[0], tuple) else (results, list(seq))\n'
    )
for _k in list(_sys.modules):
    if 'mpiplus' in _k:
        del _sys.modules[_k]

!pip install -q scipy matplotlib pandas requests gemmi

# Verify critical imports
import importlib
all_ok = True
for pkg in ['openmm', 'openmmtools', 'mdtraj', 'pymbar', 'MDAnalysis']:
    try:
        m = importlib.import_module(pkg)
        print(f"\u2713 {pkg} {getattr(m, '__version__', '')}")
    except ImportError as e:
        print(f"\u2717 {pkg}: {e}")
        all_ok = False

if all_ok:
    print("\n\u2705 All packages installed successfully!")
else:
    raise RuntimeError("Package installation failed \u2014 check error messages above")

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive', force_remount=True)
import sys, os, shutil, json, zipfile, logging, time
from pathlib import Path
import numpy as np

PIPELINE_ROOT = Path('/content/drive/MyDrive/spink7_pipeline')
assert PIPELINE_ROOT.exists()
sys.path.insert(0, str(PIPELINE_ROOT))

EXP_ID = 'EXP-30'
DRIVE_OUTPUT = Path(f'/content/drive/MyDrive/v3_gpu_results/{EXP_ID}')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
WORK_DIR = Path(f'/content/{EXP_ID}_work')
for d in ['prep','fep_complex','fep_free','analysis','figures']:
    (WORK_DIR/d).mkdir(parents=True, exist_ok=True)
logging.basicConfig(level=logging.INFO)

# ── Console log capture ──────────────────────────────────────
import io as _io
_log_buf = _io.StringIO()
_orig_stdout_write = sys.stdout.write
_orig_stderr_write = sys.stderr.write
def _stdout_log_write(data):
    _orig_stdout_write(data)
    _log_buf.write(data)
def _stderr_log_write(data):
    _orig_stderr_write(data)
    _log_buf.write(data)
sys.stdout.write = _stdout_log_write
sys.stderr.write = _stderr_log_write
# ─────────────────────────────────────────────────────────────

In [ ]:
# Checkpoint & auto-save utilities
import json, shutil, threading, time as _time
from pathlib import Path

class ExperimentCheckpoint:
    """Drive-backed phase checkpoint for resilient experiment execution."""

    def __init__(self, drive_output: Path, exp_id: str):
        self.drive_output = drive_output
        self.exp_id = exp_id
        self.path = drive_output / 'experiment_checkpoint.json'
        self.state = self._load()

    def _load(self) -> dict:
        if self.path.exists():
            with open(self.path) as f:
                return json.load(f)
        return {'exp_id': self.exp_id, 'phases': {}}

    def save(self):
        self.drive_output.mkdir(parents=True, exist_ok=True)
        with open(self.path, 'w') as f:
            json.dump(self.state, f, indent=2, default=str)

    def is_done(self, phase: str) -> bool:
        return phase in self.state['phases']

    def mark_done(self, phase: str, data: dict = None):
        self.state['phases'][phase] = {
            'timestamp': _time.strftime('%Y-%m-%d %H:%M:%S'),
            'data': data or {},
        }
        self.save()

    def get_data(self, phase: str) -> dict:
        return self.state['phases'].get(phase, {}).get('data', {})

    def summary(self):
        n = len(self.state['phases'])
        print(f'Checkpoint: {self.exp_id} — {n} phase(s) completed')
        for phase, info in self.state['phases'].items():
            print(f'  ✓ {phase} ({info["timestamp"]})')

def start_periodic_sync(work_dir: Path, drive_output: Path, interval_s: int = 300):
    """Background thread that syncs checkpoint/manifest files to Drive every N seconds."""
    stop_event = threading.Event()
    sync_patterns = ['*.chk', '*.json', '*manifest*', '*.npz', '*.npy']
    def _sync():
        while not stop_event.is_set():
            stop_event.wait(interval_s)
            if stop_event.is_set():
                break
            for pat in sync_patterns:
                for f in work_dir.rglob(pat):
                    if f.is_file() and f.stat().st_size < 500_000_000:
                        dest = drive_output / f.relative_to(work_dir)
                        dest.parent.mkdir(parents=True, exist_ok=True)
                        try:
                            shutil.copy2(f, dest)
                        except Exception:
                            pass
    t = threading.Thread(target=_sync, daemon=True, name='drive-sync')
    t.start()
    return stop_event

def restore_from_drive(drive_output: Path, work_dir: Path):
    """On session restart, copy checkpoint/manifest files from Drive back to local."""
    restored = 0
    for pat in ['*.chk', '*manifest*', '*.json']:
        for f in drive_output.rglob(pat):
            if f.is_file():
                dest = work_dir / f.relative_to(drive_output)
                dest.parent.mkdir(parents=True, exist_ok=True)
                if not dest.exists():
                    shutil.copy2(f, dest)
                    restored += 1
    if restored:
        print(f'Restored {restored} checkpoint/manifest files from Drive')

# Initialize
ckpt = ExperimentCheckpoint(DRIVE_OUTPUT, EXP_ID)
restore_from_drive(DRIVE_OUTPUT, WORK_DIR)
sync_stop = start_periodic_sync(WORK_DIR, DRIVE_OUTPUT, interval_s=300)
ckpt.summary()
print('Auto-save: checkpoint/manifest files sync to Drive every 5 min')

In [ ]:
import openmm
from openmm import unit, Platform
Platform.getPlatformByName('CUDA')
print(f'OpenMM {openmm.__version__}, CUDA ready')

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats as sp_stats
from src.config import (SystemConfig, MinimizationConfig, EquilibrationConfig,
                        ProductionConfig, FEPConfig, KCAL_TO_KJ)
from src.prep.pdb_fetch import fetch_pdb
from src.prep.pdb_clean import clean_structure
from src.prep.topology import build_topology
from src.prep.solvate import solvate_system
from src.simulate.minimizer import minimize_energy
from src.simulate.equilibrate import run_nvt, run_npt
from src.simulate.fep import run_fep_campaign
from src.simulate.platform import select_platform
from src.analyze.fep import compute_delta_g_mbar, compute_delta_delta_g
from openmm.app import PME, ForceField, Simulation, HBonds, PDBFile
from openmm import LangevinMiddleIntegrator, XmlSerializer
import mdtraj as md
print('Imports OK')

In [ ]:
TEMPERATURE_K = 310.0
system_config = SystemConfig()
min_config = MinimizationConfig(max_iterations=10000)
eq_config = EquilibrationConfig(nvt_duration_ps=500.0, npt_duration_ps=1000.0, temperature_k=TEMPERATURE_K)
fep_config = FEPConfig(n_lambda_windows=16, per_window_duration_ns=2.0, temperature_k=TEMPERATURE_K)  # optimized from 20: sufficient MBAR overlap for X→Ala

# Experimental DDG values (kcal/mol, positive = destabilizing)
MUTATIONS = {
    'K15A': {'resid': 15, 'ddg_exp': 10.0},
    'R17A': {'resid': 17, 'ddg_exp': 3.5},
    'I18A': {'resid': 18, 'ddg_exp': 2.0},
    'I19A': {'resid': 19, 'ddg_exp': 2.5},
    'R20A': {'resid': 20, 'ddg_exp': 1.5},
    'Y21A': {'resid': 21, 'ddg_exp': 1.8},
    'F22A': {'resid': 22, 'ddg_exp': 1.0},
    'Y23A': {'resid': 23, 'ddg_exp': 2.3},
    'N24A': {'resid': 24, 'ddg_exp': 0.8},
    'C14A': {'resid': 14, 'ddg_exp': 3.0},
    'P13A': {'resid': 13, 'ddg_exp': 1.2},
    'G12A': {'resid': 12, 'ddg_exp': 0.0},  # Gly->Ala is a gain
    'T11A': {'resid': 11, 'ddg_exp': 0.5},
    'G36A': {'resid': 36, 'ddg_exp': 0.3},
    'G37A': {'resid': 37, 'ddg_exp': 0.2},
}
print(f'{len(MUTATIONS)} mutations to scan')

In [ ]:
# Helper: get sidechain atom indices for a residue (atoms beyond CA, C, N, O, CB, H, HA)
BACKBONE_NAMES = {'N', 'CA', 'C', 'O', 'H', 'HA', 'HA2', 'HA3', 'OXT'}

def get_sidechain_indices(topology, resSeq, chain_id=None):
    """Get atom indices of sidechain heavy atoms beyond CB for a given residue.
    For Gly (no sidechain), returns empty. For Ala, returns CB only.
    For mutation to Ala, we want atoms to annihilate = sidechain beyond CB."""
    indices = []
    for atom in topology.atoms:
        if atom.residue.resSeq == resSeq:
            if chain_id is not None and atom.residue.chain.index != chain_id:
                continue
            if atom.name not in BACKBONE_NAMES and atom.name != 'CB':
                indices.append(atom.index)
    return indices

def get_all_residue_indices(topology, resSeq, chain_id=None):
    """Get all atom indices for a residue."""
    indices = []
    for atom in topology.atoms:
        if atom.residue.resSeq == resSeq:
            if chain_id is not None and atom.residue.chain.index != chain_id:
                continue
            indices.append(atom.index)
    return indices

print('Helper functions defined')

In [ ]:
# Build or load COMPLEX system (BPTI-Trypsin from 2PTC)
PREP_DIR = WORK_DIR / 'prep'
EXP04_DIR = Path('/content/drive/MyDrive/v3_gpu_results/EXP-04')

# Try to load pre-equilibrated system from EXP-04
exp04_system_xml = EXP04_DIR / 'system.xml'
exp04_state_xml = EXP04_DIR / 'eq_state.xml'
if not exp04_state_xml.exists():
    exp04_state_xml = list(EXP04_DIR.rglob('npt_final_state.xml'))
    exp04_state_xml = exp04_state_xml[0] if exp04_state_xml else None

if exp04_system_xml.exists() and exp04_state_xml and Path(exp04_state_xml).exists():
    print('Loading pre-equilibrated complex from EXP-04...')
    complex_system = XmlSerializer.deserialize(exp04_system_xml.read_text())
    complex_state = XmlSerializer.deserialize(Path(exp04_state_xml).read_text())
    complex_positions = complex_state.getPositions(asNumpy=True)
    # Load topology PDB for atom selection
    topo_candidates = list(EXP04_DIR.rglob('topology_reference.pdb')) + list(EXP04_DIR.rglob('*solvated*.pdb'))
    complex_topo_pdb = str(topo_candidates[0]) if topo_candidates else None
    print(f'Complex loaded from EXP-04')
else:
    print('Building complex system from 2PTC...')
    raw_pdb = fetch_pdb('2PTC', output_dir=PREP_DIR)
    cleaned = clean_structure(raw_pdb, chains_to_keep=['E', 'I'], remove_heteroatoms=True, remove_waters=True)
    _, complex_system, modeller = build_topology(cleaned, system_config)
    modeller, n_water, _, _ = solvate_system(modeller, system_config)
    ff = ForceField(system_config.force_field, system_config.water_model)
    complex_system = ff.createSystem(modeller.topology, nonbondedMethod=PME,
        nonbondedCutoff=1.0*unit.nanometer, constraints=HBonds, rigidWater=True)
    integrator = LangevinMiddleIntegrator(TEMPERATURE_K*unit.kelvin, 1.0/unit.picosecond, 0.002*unit.picoseconds)
    sim = Simulation(modeller.topology, complex_system, integrator, select_platform('CUDA'))
    sim.context.setPositions(modeller.positions)
    complex_topo_pdb = str(PREP_DIR / 'complex_solvated.pdb')
    with open(complex_topo_pdb, 'w') as f:
        PDBFile.writeFile(modeller.topology, modeller.positions, f)
    minimize_energy(sim, min_config)
    run_nvt(sim, eq_config, WORK_DIR/'fep_complex'/'nvt')
    run_npt(sim, eq_config, WORK_DIR/'fep_complex'/'npt')
    complex_state = sim.context.getState(getPositions=True)
    complex_positions = complex_state.getPositions(asNumpy=True)
    with open(WORK_DIR/'fep_complex'/'system.xml', 'w') as f:
        f.write(XmlSerializer.serialize(complex_system))
    print(f'Complex equilibrated: {n_water} waters')

In [ ]:
# Build FREE-leg system (isolated BPTI in water)
raw_bpti = fetch_pdb('4PTI', output_dir=PREP_DIR)
cleaned_bpti = clean_structure(raw_bpti, chains_to_keep=['I'], remove_heteroatoms=True, remove_waters=True)
_, free_system, modeller_free = build_topology(cleaned_bpti, system_config)
modeller_free, n_water_free, _, _ = solvate_system(modeller_free, system_config)
ff = ForceField(system_config.force_field, system_config.water_model)
free_system = ff.createSystem(modeller_free.topology, nonbondedMethod=PME,
    nonbondedCutoff=1.0*unit.nanometer, constraints=HBonds, rigidWater=True)
integrator_free = LangevinMiddleIntegrator(TEMPERATURE_K*unit.kelvin, 1.0/unit.picosecond, 0.002*unit.picoseconds)
sim_free = Simulation(modeller_free.topology, free_system, integrator_free, select_platform('CUDA'))
sim_free.context.setPositions(modeller_free.positions)

free_topo_pdb = str(PREP_DIR / 'bpti_free_solvated.pdb')
with open(free_topo_pdb, 'w') as f:
    PDBFile.writeFile(modeller_free.topology, modeller_free.positions, f)

minimize_energy(sim_free, min_config)
run_nvt(sim_free, eq_config, WORK_DIR/'fep_free'/'nvt')
run_npt(sim_free, eq_config, WORK_DIR/'fep_free'/'npt')
free_state = sim_free.context.getState(getPositions=True)
free_positions = free_state.getPositions(asNumpy=True)
with open(WORK_DIR/'fep_free'/'system.xml', 'w') as f:
    f.write(XmlSerializer.serialize(free_system))
print(f'Free BPTI equilibrated: {n_water_free} waters')

In [ ]:
# Load topologies for atom index selection
complex_top = md.load(complex_topo_pdb).topology if complex_topo_pdb else None
free_top = md.load(free_topo_pdb).topology

# For the complex, BPTI is chain 1 (index 1); for free BPTI, it's chain 0
# Adjust chain_id based on actual chain ordering
complex_chains = list(complex_top.chains) if complex_top else []
free_chains = list(free_top.chains)

# Identify which chain is BPTI in complex (smaller chain by atom count)
if complex_top:
    chain_atoms = [(c.index, len(list(c.atoms))) for c in complex_chains]
    bpti_chain_complex = min(chain_atoms, key=lambda x: x[1])[0]  # BPTI is smaller
else:
    bpti_chain_complex = 1
bpti_chain_free = 0  # only chain in free system

print(f'BPTI chain in complex: {bpti_chain_complex}')
print(f'BPTI chain in free: {bpti_chain_free}')

In [ ]:
# Run FEP for each mutation
ddg_results = {}
checkpoint_file = DRIVE_OUTPUT / 'checkpoint_ddg.json'

# Resume from checkpoint if available
if checkpoint_file.exists():
    with open(checkpoint_file) as f:
        ddg_results = json.load(f)
    print(f'Resumed {len(ddg_results)} mutations from checkpoint')

for mut_name, mut_info in MUTATIONS.items():
    if mut_name in ddg_results:
        print(f'{mut_name}: already computed, skipping')
        continue

    resid = mut_info['resid']
    t_start = time.time()
    print(f'\n=== {mut_name} (resid {resid}) ===')

    # Get sidechain atom indices in complex system
    if complex_top:
        sc_complex = get_sidechain_indices(complex_top, resid, chain_id=bpti_chain_complex)
    else:
        sc_complex = []
    sc_free = get_sidechain_indices(free_top, resid, chain_id=bpti_chain_free)

    # For Gly or Ala (no sidechain beyond CB), use all residue atoms
    if not sc_complex and complex_top:
        sc_complex = get_all_residue_indices(complex_top, resid, chain_id=bpti_chain_complex)
    if not sc_free:
        sc_free = get_all_residue_indices(free_top, resid, chain_id=bpti_chain_free)

    print(f'  Complex sidechain atoms: {len(sc_complex)}')
    print(f'  Free sidechain atoms: {len(sc_free)}')

    if not sc_complex or not sc_free:
        print(f'  SKIP: no atoms found')
        continue

    # Complex leg FEP
    complex_out = WORK_DIR / 'fep_complex' / mut_name
    fep_complex = run_fep_campaign(
        system=complex_system, positions=complex_positions,
        mutant_atom_indices=sc_complex, config=fep_config, output_dir=complex_out)
    dg_complex = compute_delta_g_mbar(
        fep_complex['energy_matrix'], fep_complex['n_samples_per_state'], TEMPERATURE_K)
    print(f'  Complex \u0394G: {dg_complex["delta_g_kcal_mol"]:.2f} \u00b1 {dg_complex["delta_g_std_kcal_mol"]:.2f}')

    # Free leg FEP
    free_out = WORK_DIR / 'fep_free' / mut_name
    fep_free = run_fep_campaign(
        system=free_system, positions=free_positions,
        mutant_atom_indices=sc_free, config=fep_config, output_dir=free_out)
    dg_free = compute_delta_g_mbar(
        fep_free['energy_matrix'], fep_free['n_samples_per_state'], TEMPERATURE_K)
    print(f'  Free \u0394G: {dg_free["delta_g_kcal_mol"]:.2f} \u00b1 {dg_free["delta_g_std_kcal_mol"]:.2f}')

    # DDG
    ddg = compute_delta_delta_g(dg_complex, dg_free)
    elapsed = time.time() - t_start
    print(f'  \u0394\u0394G: {ddg["delta_delta_g_kcal_mol"]:.2f} \u00b1 {ddg["delta_delta_g_std_kcal_mol"]:.2f} kcal/mol ({elapsed:.0f}s)')

    ddg_results[mut_name] = {
        'ddg_sim': ddg['delta_delta_g_kcal_mol'], 'ddg_std': ddg['delta_delta_g_std_kcal_mol'],
        'dg_complex': dg_complex['delta_g_kcal_mol'], 'dg_free': dg_free['delta_g_kcal_mol'],
        'ddg_exp': mut_info['ddg_exp'], 'elapsed_s': elapsed,
    }

    # Checkpoint after each mutation
    with open(checkpoint_file, 'w') as f:
        json.dump(ddg_results, f, indent=2)

print(f'\nCompleted {len(ddg_results)} mutations')

In [ ]:
# Correlation analysis
muts_computed = [m for m in MUTATIONS if m in ddg_results]
sim_vals = np.array([ddg_results[m]['ddg_sim'] for m in muts_computed])
exp_vals = np.array([ddg_results[m]['ddg_exp'] for m in muts_computed])
std_vals = np.array([ddg_results[m]['ddg_std'] for m in muts_computed])

r_val, p_val = sp_stats.pearsonr(exp_vals, sim_vals)
rmse = np.sqrt(np.mean((sim_vals - exp_vals)**2))

# Sign agreement
sign_correct = np.sum(np.sign(sim_vals) == np.sign(exp_vals))
sign_pct = sign_correct / len(sim_vals) * 100

print(f'Pearson R: {r_val:.3f} (p={p_val:.4e})')
print(f'RMSE: {rmse:.2f} kcal/mol')
print(f'Sign agreement: {sign_correct}/{len(sim_vals)} ({sign_pct:.0f}%)')

if r_val > 0.7 and rmse < 2.0 and sign_pct >= 80:
    verdict = 'PASS'
elif r_val > 0.5 and rmse < 3.0:
    verdict = 'MARGINAL'
else:
    verdict = 'FAIL'
print(f'\nVERDICT: {verdict}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.errorbar(exp_vals, sim_vals, yerr=std_vals, fmt='o', color='steelblue',
            ecolor='lightgray', elinewidth=1, capsize=3, markersize=8, zorder=5)
for i, m in enumerate(muts_computed):
    ax.annotate(m, (exp_vals[i], sim_vals[i]), textcoords='offset points',
                xytext=(6, 5), fontsize=7)

lims = [min(min(exp_vals), min(sim_vals))-1, max(max(exp_vals), max(sim_vals))+1]
ax.plot(lims, lims, 'k--', alpha=0.3, label='y=x')
slope, intercept = np.polyfit(exp_vals, sim_vals, 1)
x_fit = np.linspace(lims[0], lims[1], 100)
ax.plot(x_fit, slope*x_fit + intercept, 'r-', alpha=0.6,
        label=f'R={r_val:.2f}, RMSE={rmse:.1f}')

ax.set_xlabel('Experimental \u0394\u0394G (kcal/mol)'); ax.set_ylabel('Computed \u0394\u0394G (kcal/mol)')
ax.set_title(f'EXP-30: Alanine Scanning \u2014 {verdict}'); ax.legend()
ax.set_aspect('equal'); fig.tight_layout()
fig.savefig(WORK_DIR/'figures'/'results.png', dpi=150); plt.close(fig)

In [ ]:
sync_stop.set()  # Stop periodic sync before final copy

results = {
    'experiment_id': EXP_ID, 'pearson_r': round(float(r_val), 4),
    'rmse_kcal': round(float(rmse), 3), 'sign_agreement_pct': round(sign_pct, 1),
    'n_mutations': len(muts_computed), 'verdict': verdict,
    'per_mutation': ddg_results,
}
with open(WORK_DIR/'results.json', 'w') as f: json.dump(results, f, indent=2)

for item in WORK_DIR.rglob('*'):
    if item.is_file():
        dest = DRIVE_OUTPUT / item.relative_to(WORK_DIR)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(item, dest)

ckpt.mark_done('complete')

# ── Save full console log ────────────────────────────────────
_console_log_text = _log_buf.getvalue()
(WORK_DIR / 'console_log.txt').write_text(_console_log_text)
(DRIVE_OUTPUT / 'console_log.txt').write_text(_console_log_text)
print(f'Console log saved: {len(_console_log_text)} chars')
# ─────────────────────────────────────────────────────────────
zip_path = Path(f'/content/{EXP_ID}_results.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for item in WORK_DIR.rglob('*'):
        if item.is_file(): zf.write(item, f'{EXP_ID}/{item.relative_to(WORK_DIR)}')
files.download(str(zip_path))